# TUTOR-60 · 5 — The recommendation

A fitted curve is not a decision. The state has nine million dollars, twelve thousand
eligible students, and a choice with two dimensions and a hard budget line. This notebook
turns the posterior into that choice, says how sure it is, says what would change it, and
writes it down in a form somebody can act on.

| | |
|---|---|
| **1** | the objective, evaluated through the posterior rather than at its mean |
| **2** | buy the cheap thing first — `surface.allocate` and the budget frontier |
| **3** | the recommendation, and the paired comparison against the pilot |
| **4** | what would change it: the response family, the budget, the price of a point |
| **5** | the memo, the ledger, and an analysis that reloads |

In [ ]:
import sys

sys.path.insert(0, ".")

import pathlib
import tempfile

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import tutoring as T
from axiom.core import LedgerLine, Posterior, is_failure, summarize
from axiom.diagnose import TippingPoint, tipping_point
from axiom.infer import PymcBackend
from axiom.io import Analysis, load_analysis, save_analysis
from axiom.report import ReportBuilder, render_html, resolve, write
from axiom.surface import (
    Allocation, Frontier, HillKernel, Surface, SplineKernel, allocate, fit, frontier,
    marginal_band, predict, response_band,
)

program = T.run(T.chosen_design())
spline_spec = T.planning_spec()
result = fit(spline_spec, T.school_panel(program), backend=PymcBackend(nuts_sampler="nutpie"),
             draws=1000, tune=1000, chains=4, seed=0)
surface = Surface(spline_spec)
print(f"budget {T.BUDGET:,.0f} USD | {T.N_ELIGIBLE:,} eligible | "
      f"{T.BUDGET / T.N_ELIGIBLE:,.0f} USD a student if the state reaches all of them")

## 1 · The objective, through the posterior

```
cohort points = min(eligible, budget / cost_per_student) x growth(cost_per_student)
```

The second factor is a posterior quantity, so the whole objective is. Evaluating it at
the posterior *mean* of the parameters would be a different number — the objective is
nonlinear in the response and the response is nonlinear in the parameters — so every
draw goes through the same `forward` the likelihood used.

In [ ]:
tutoring_grid = np.arange(0.0, T.TUTORING_MAX + 1.0, 25.0)
messaging_grid = np.asarray(T.MESSAGING_LEVELS)
TUT, MSG = np.meshgrid(tutoring_grid, messaging_grid, indexing="ij")

def effect_draws(fitted, spec_):
    """Posterior draws of program effect at every point of the allocation grid."""
    surf = Surface(spec_)
    nothing = predict(surf, fitted.posterior, {"tutoring": np.zeros_like(TUT),
                                               "messaging": np.zeros_like(MSG)}, seed=0)
    something = predict(surf, fitted.posterior, {"tutoring": TUT, "messaging": MSG}, seed=0)
    return (np.asarray(something.values) - np.asarray(nothing.values)).reshape(-1, *TUT.shape)

effect = effect_draws(result, spline_spec)
served = T.students_served(TUT, MSG)
cohort = served[None, :, :] * effect
print(f"{cohort.shape[0]:,} posterior draws x {TUT.shape[0]} tutoring levels x "
      f"{TUT.shape[1]} messaging levels")

expected = cohort.mean(axis=0)
best = np.unravel_index(int(np.argmax(expected)), expected.shape)
recommended = (float(tutoring_grid[best[0]]), float(messaging_grid[best[1]]))
print(f"\nhighest expected cohort points: tutoring {recommended[0]:,.0f} USD "
      f"({recommended[0] / T.MINUTE_COST:.0f} weekly minutes) + messaging {recommended[1]:,.0f} USD "
      f"({recommended[1] / T.MESSAGE_COST:.0f} messages a week)")
print(f"  reaches {float(served[best]):,.0f} of {T.N_ELIGIBLE:,} eligible students")
print(f"  expected {expected[best]:,.0f} cohort points")

## 2 · Buy the cheap thing first

Before the level of spending, the *split*. `surface.allocate` maximizes the fitted
response subject to a per-student budget; `surface.frontier` runs it across budgets and
reports the shadow price of the constraint.

In [ ]:
budgets = [200.0, 400.0, 600.0, 750.0, 1000.0, 1500.0, 2000.0, 3000.0]
front = frontier(surface, result.posterior, budgets=budgets, bounds=T.bounds(), seed=0)
assert isinstance(front, Frontier)
table = front.as_frame()
table["weekly minutes"] = table["tutoring"] / T.MINUTE_COST
table["messages/wk"] = table["messaging"] / T.MESSAGE_COST
print(table.round(2).to_string(index=False))

at_750 = allocate(surface, result.posterior, budget=T.BUDGET / T.N_ELIGIBLE,
                  bounds=T.bounds(), seed=0)
assert isinstance(at_750, Allocation)
print(f"\nat {T.BUDGET / T.N_ELIGIBLE:,.0f} USD a student: {at_750.doses} ({at_750.status})")

Messaging is pinned at its cap — 45 dollars, five messages a week — at **every** budget
from 200 dollars up, and everything above that goes to tutoring. That is not a subtle
result and it does not need a posterior to see once the numbers are on one scale: five
messages a week buy about 1.4 points for 45 dollars; the same 45 dollars of tutoring buy
roughly a tenth of a point.

Note the last row. At 3,000 dollars a student the optimizer stops spending at about
2,100 — past that the fitted response is falling, so the extra money has nowhere useful
to go. A monotone response family cannot produce that row.

In [ ]:
fig = T.figure("What a per-student budget buys, and how it splits",
               "per-student budget (USD/year)", f"expected growth ({T.OUTCOME_UNIT})", height=420)
fig.add_trace(go.Scatter(x=table.budget, y=table.expected_outcome, name="expected growth",
                         line={"color": T.TUTORING_COLOR, "width": 3}))
fig.add_trace(go.Bar(x=table.budget, y=table.messaging, name="to messaging",
                     marker_color=T.rgba(T.MESSAGING_COLOR, 0.5), yaxis="y2", width=90))
fig.update_layout(yaxis2={"title": "USD to messaging", "overlaying": "y", "side": "right",
                          "range": [0, T.MESSAGING_MAX * 3], "showgrid": False})
fig.add_vline(x=T.BUDGET / T.N_ELIGIBLE, line={"dash": "dot", "color": T.DECISION_COLOR},
              annotation_text="750 USD — everyone served")
fig.show()

## 3 · The recommendation, and how sure it is

The allocation above maximizes growth *per student*. The state's objective is points
across the cohort, so the level of spending is set by where the budget stops reaching
everybody. That trade is what the next figure is.

In [ ]:
at_cap = messaging_grid == T.MESSAGING_MAX
curve = cohort[:, :, at_cap].squeeze(-1)
mean_curve = curve.mean(axis=0)
lower, upper = np.percentile(curve, [5, 95], axis=0)
within_two_percent = tutoring_grid[mean_curve >= 0.98 * mean_curve.max()]

fig = T.figure("Total reading points the program delivers, against what it spends a student",
               "tutoring", "cohort reading points", height=460)
T.band(fig, tutoring_grid, lower, upper, T.TUTORING_COLOR, name="90% posterior band")
fig.add_trace(go.Scatter(x=tutoring_grid, y=mean_curve, name="posterior mean",
                         line={"color": T.TUTORING_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=tutoring_grid,
                         y=T.students_served(tutoring_grid, T.MESSAGING_MAX)
                           * T.mean_gain(tutoring_grid * T.TRUTH.fidelity_mean, T.MESSAGING_MAX),
                         name="the truth (synthetic world)",
                         line={"color": T.TRUTH_COLOR, "width": 2, "dash": "dot"}))
fig.add_vrect(x0=float(within_two_percent.min()), x1=float(within_two_percent.max()),
              fillcolor=T.rgba(T.MESSAGING_COLOR, 0.16), line_width=0,
              annotation_text="within 2% of the best", annotation_position="top left")
fig.add_vline(x=1800.0, line={"color": T.DECISION_COLOR, "width": 2},
              annotation_text="the pilot's 90 minutes", annotation_position="top right")
T.minutes_axis(fig)
fig.show()
print(f"allocations within 2% of the best: {within_two_percent.min():,.0f}-"
      f"{within_two_percent.max():,.0f} USD "
      f"({within_two_percent.min() / T.MINUTE_COST:.0f}-{within_two_percent.max() / T.MINUTE_COST:.0f} "
      f"weekly minutes)")

In [ ]:
flat = cohort.reshape(cohort.shape[0], -1)
argmax = np.argmax(flat, axis=1)
best_tutoring, best_messaging = np.unravel_index(argmax, TUT.shape)
print("posterior over the recommended allocation:")
print(f"  tutoring : median {np.median(tutoring_grid[best_tutoring]):,.0f} USD, "
      f"5-95% [{np.percentile(tutoring_grid[best_tutoring], 5):,.0f}, "
      f"{np.percentile(tutoring_grid[best_tutoring], 95):,.0f}]")
print(f"  messaging at the cap in {(messaging_grid[best_messaging] == T.MESSAGING_MAX).mean():.0%} of draws")

pilot_index = (int(np.argmin(np.abs(tutoring_grid - 1800.0))), int(np.argmax(at_cap)))
difference = cohort[:, best[0], best[1]] - cohort[:, pilot_index[0], pilot_index[1]]
summary = summarize(difference, mass=0.9)
print(f"\nrecommended allocation minus the pilot's 90 minutes, paired within each draw:")
print(f"  {summary.mean:,.0f} cohort points {summary.interval}")
print(f"  P(the recommendation is better) = {float((difference > 0).mean()):.3f}")
print(f"  worth {summary.mean * T.VALUE_PER_POINT:,.0f} USD at {T.VALUE_PER_POINT:,.0f} USD a point")
print(f"  the truth, for the record: "
      f"{float(T.cohort_points(recommended[0] * T.TRUTH.fidelity_mean, recommended[1]) - T.cohort_points(1800.0 * T.TRUTH.fidelity_mean, T.MESSAGING_MAX)):,.0f}")

In [ ]:
fig = T.figure("The recommendation against the pilot, paired within every posterior draw",
               "extra cohort reading points", "density", height=400)
counts, edges = np.histogram(difference, bins=60, density=True)
fig.add_trace(go.Scatter(x=0.5 * (edges[1:] + edges[:-1]), y=counts, fill="tozeroy",
                         fillcolor=T.rgba(T.MESSAGING_COLOR, 0.25),
                         line={"color": T.MESSAGING_COLOR, "width": 2}, name="posterior"))
fig.add_vline(x=0.0, line={"color": T.DECISION_COLOR, "width": 2},
              annotation_text="the pilot is better to the left of this line")
fig.show()

## 4 · What would change it

Three levers, in the order they matter.

### The response family

Notebook 3 showed the monotone Hill fit is indistinguishable from the spline on every
posterior predictive check and says something different about the top of the range.
Here is what that is worth.

In [ ]:
hill_spec = T.spec({"tutoring": HillKernel(reference_dose=900.0, amplitude_scale=8.0),
                    "messaging": SplineKernel(reference_dose=T.MESSAGING_MAX, amplitude_scale=2.0,
                                              knots=T.MESSAGING_KNOTS)}, name="tutor60_hill")
hill = fit(hill_spec, T.school_panel(program),
           backend=PymcBackend(nuts_sampler="nutpie", target_accept=0.95),
           draws=1500, tune=2000, chains=4, seed=0)
hill_cohort = served[None, :, :] * effect_draws(hill, hill_spec)

rows = []
for multiplier in (1.0, 2.0, 3.0):
    budget = T.BUDGET * multiplier
    scale = T.students_served(TUT, MSG, budget=budget) / np.maximum(served, 1e-9)
    for label, block in (("spline", cohort), ("Hill", hill_cohort)):
        totals = (block * scale[None, :, :]).mean(axis=0)
        where = np.unravel_index(int(np.argmax(totals)), totals.shape)
        rows.append({"budget (M USD)": budget / 1e6, "family": label,
                     "tutoring (USD)": tutoring_grid[where[0]],
                     "weekly minutes": tutoring_grid[where[0]] / T.MINUTE_COST,
                     "cohort points": totals[where]})
    truth = T.students_served(TUT, MSG, budget=budget) * T.mean_gain(TUT * T.TRUTH.fidelity_mean, MSG)
    where = np.unravel_index(int(np.argmax(truth)), truth.shape)
    rows.append({"budget (M USD)": budget / 1e6, "family": "the truth",
                 "tutoring (USD)": tutoring_grid[where[0]],
                 "weekly minutes": tutoring_grid[where[0]] / T.MINUTE_COST,
                 "cohort points": truth[where]})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

At the budget the state actually has, the two families **recommend the same allocation**
and disagree by about a sixth on what it will deliver. That is worth knowing and it is
not what decides anything: the budget line is doing more work than the response family.

Triple the budget and they part company, because the decision moves to the part of the
curve where one family can turn over and the other cannot. The family choice is a bet on
a region of the dose range the state is not currently spending in — which is the honest
way to describe most functional-form assumptions.

### How wrong would the estimate have to be

`diagnose.tipping_point` asks the question a sceptical board asks: how much would the
effect have to be overstated before the recommendation flips back to the pilot?

In [ ]:
grid = np.linspace(0.0, 2.0, 101) * float(difference.mean())
tipping: TippingPoint = tipping_point(difference, decision_threshold=0.0, bias_grid=grid,
                                      name="recommendation minus pilot", certainty=0.5)
print(f"posterior mean advantage       {float(difference.mean()):>12,.0f} cohort points")
print(f"decision at zero bias          {tipping.decision_at_zero!s:>12s} "
      f"(P = {tipping.probability_at_zero:.3f})")
if tipping.flipped and tipping.bias is not None:
    print(f"overstatement that flips it    {tipping.bias:>12,.0f} cohort points")
    print(f"  as a share of the advantage  {tipping.bias / float(difference.mean()):>12.1%}")
    print(f"  at that bias P(better) =     {tipping.probability:>12.3f}, {tipping.interval}")
else:
    print("  the recommendation does not flip anywhere on this grid")

fig = T.figure("How much the advantage would have to be overstated",
               "assumed overstatement (cohort points)",
               "P(the recommendation still beats the pilot)", height=380)
fig.add_trace(go.Scatter(x=tipping.bias_grid, y=tipping.probabilities,
                         line={"color": T.DECISION_COLOR, "width": 3}, name="posterior probability"))
fig.add_hline(y=tipping.certainty, line={"dash": "dot", "color": T.TRUTH_COLOR},
              annotation_text=f"decision flips below {tipping.certainty:.0%}")
if tipping.bias is not None:
    fig.add_vline(x=tipping.bias, line={"color": T.ACCENT, "width": 2},
                  annotation_text=f"{tipping.bias:,.0f} points")
fig.show()

### The price of a point, and the size of the cohort

The dollar value of a reading point does not change the recommendation — it scales the
objective without moving its argmax — but it decides whether the program is worth funding
at all. The **eligible count** does move the recommendation, because it moves the point
at which the budget stops reaching everybody.

In [ ]:
print(f"{'eligible students':>18s} {'recommended tutoring':>22s} {'weekly minutes':>16s} {'reached':>10s}")
for n_eligible in (6_000, 9_000, 12_000, 18_000, 24_000):
    reach = T.students_served(TUT, MSG, n_eligible=n_eligible)
    totals = (cohort / np.maximum(served, 1e-9)[None] * reach[None]).mean(axis=0)
    where = np.unravel_index(int(np.argmax(totals)), totals.shape)
    print(f"{n_eligible:>18,d} {tutoring_grid[where[0]]:>18,.0f} USD "
          f"{tutoring_grid[where[0]] / T.MINUTE_COST:>16.0f} {float(reach[where]):>10,.0f}")
print(f"\nvalue of the recommendation over the pilot, at three prices for a reading point:")
for price in (700.0, T.VALUE_PER_POINT, 2_800.0):
    print(f"  {price:>7,.0f} USD/point -> {summary.mean * price:>14,.0f} USD "
          f"[{summary.interval.lower * price:,.0f}, {summary.interval.upper * price:,.0f}]")

## 5 · The recommendation, written down

Same template language as notebook 4, one more section, and the numbers this notebook
computed. A recommendation that cannot be regenerated next June from a new fit is a
document; this one is a function.

In [ ]:
band = response_band(result, "tutoring", n_grid=41, mass=0.9)
slope = marginal_band(result, "tutoring", n_grid=41, mass=0.9)
memo = (
    ReportBuilder("tutor60_recommendation", "TUTOR-60 — recommendation to the board")
    .subtitle("Prepared {as_of} from the {n_schools}-school trial. Intervals are {interval_label}.")
    .footer("axiom.report · fit {fit_hash} · every number regenerates from the posterior")
    .section("Recommendation",
             summary="Fund 35 weekly minutes of tutoring and five caregiver messages a week, "
                     "for the whole eligible cohort.")
    .paragraph("Allocate **{tutoring_usd:,.0f} dollars** a student to tutoring "
               "({minutes:.0f} weekly minutes) and **{messaging_usd:,.0f} dollars** to caregiver "
               "messaging ({messages:.0f} a week). That reaches {reached:,.0f} of the "
               "{eligible:,.0f} eligible students and is expected to deliver "
               "{cohort_points:,.0f} reading points across the cohort.")
    .paragraph("Against the pilot district's 90 weekly minutes — which the budget would "
               "stretch to only {pilot_reached:,.0f} students — the recommendation is ahead by "
               "{advantage_points:,.0f} points, and is the better choice in "
               "{probability:.0%} of the posterior.")
    .metric("advantage", "Cohort points over the pilot allocation", unit="points")
    .metric("advantage_value", "Value of the difference", unit="USD")
    .section("What it rests on")
    .figure("response", caption="Fitted growth against the tutoring allocation, 90% band")
    .figure("marginal", caption="The marginal effect crosses zero inside the range the trial "
                                "tested — spending past it is expected to do harm")
    .paragraph("The recommendation is flat between {flat_low:,.0f} and {flat_high:,.0f} dollars: "
               "anything in that range is within 2% of the best. The decision that matters is "
               "not the exact figure, it is not funding 90 minutes.")
    .ledger("ledger", caption="Assumptions, and where each number came from", show_detail=True)
    .build()
)
memo_context = {
    "as_of": "2027-06-30", "n_schools": program.n_schools, "interval_label": band.label(),
    "fit_hash": spline_spec.content_hash()[:12],
    "tutoring_usd": recommended[0], "minutes": recommended[0] / T.MINUTE_COST,
    "messaging_usd": recommended[1], "messages": recommended[1] / T.MESSAGE_COST,
    "reached": float(served[best]), "eligible": float(T.N_ELIGIBLE),
    "cohort_points": float(expected[best]),
    "pilot_reached": float(T.students_served(1800.0, T.MESSAGING_MAX)),
    "advantage_points": float(difference.mean()),
    "advantage": summarize(difference, mass=0.9),
    "advantage_value": summarize(difference * T.VALUE_PER_POINT, mass=0.9),
    "probability": float((difference > 0).mean()),
    "flat_low": float(within_two_percent.min()), "flat_high": float(within_two_percent.max()),
    "response": band, "marginal": slope,
    "ledger": [
        LedgerLine(kind="identification",
                   statement="schools were randomized to allocations; the adjustment set is empty",
                   detail={"route": "backdoor", "unit_of_assignment": "school"}),
        LedgerLine(kind="estimand",
                   statement="the recommendation is over *assigned* allocations; schools "
                             "delivered 93% of assigned minutes and that is priced in",
                   detail={"version": "assigned"}),
        LedgerLine(kind="interval",
                   statement=f"every figure and metric at {band.label()} over "
                             f"{band.n_draws:,} posterior draws"),
        LedgerLine(kind="assumption",
                   statement="a reading point is valued at 1,400 USD; the recommendation does "
                             "not depend on this number, only on whether to fund at all",
                   detail={"source": "state cost-benefit memorandum 2026"}),
        LedgerLine(kind="assumption",
                   statement="a monotone response family fitted to the same data recommends the "
                             "same allocation and a sixth more value; the families part company "
                             "only at three times this budget",
                   detail={"checked": "notebook 3 section 6"}),
        LedgerLine(kind="limitation",
                   statement="the grade-band breakdown the program office asked for is not "
                             "estimable from a school-randomized trial and is not reported",
                   detail={"blocked_estimand": "growth_at_pilot_allocation"}),
    ],
}
resolved_memo = resolve(memo, memo_context)
out = pathlib.Path(tempfile.mkdtemp())
for fmt in ("html", "pptx", "pdf"):
    written = write(memo, memo_context, str(out / f"recommendation.{fmt}"), fmt)
    print(f"{fmt:5s} -> {pathlib.Path(written).stat().st_size:>10,} bytes"
          if not is_failure(written) else f"{fmt:5s} refused: {written.reason}")
print()
for section in resolved_memo.sections:
    print(f"[{section.title}]")
    for block in section.blocks:
        text = getattr(block, "text", None)
        if text:
            print("  ", text[:150].replace("**", ""))

## 6 · The analysis, saved

The recommendation is only as reproducible as what it was computed from. `io.save_analysis`
writes the specs, the panel, the posterior and the ledger with a content hash on each and
no pickle anywhere.

In [ ]:
analysis = (
    Analysis(specs={"surface": spline_spec, "report": memo, "window": T.school_year()})
    .with_panel(T.school_panel(program))
    .with_posterior(result.posterior)
)
for line in memo_context["ledger"]:
    analysis = analysis.with_ledger_line(line)
root = pathlib.Path(tempfile.mkdtemp()) / "tutor60.axiom"
saved = save_analysis(analysis, root, seed=0)
print(saved.summary())
for name, digest in saved.hashes().items():
    print(f"  {name:12s} {digest[:16]}")
reloaded = load_analysis(root)
print("\nreloads identical:", reloaded == saved)
print("the report template came back too:",
      reloaded.spec("report").content_hash() == memo.content_hash())

## The recommendation

> **Fund 35 weekly minutes of small-group tutoring and five caregiver messages a week,
> for all 12,000 eligible students.** Not 90 minutes for the 4,900 the same money reaches.

The reasoning, in the order it was established:

1. **The objective is a cohort total.** With a fixed budget, every extra dollar per
   student is a student not served, and the response curve is steep at the bottom. The
   trade is settled at about 750 dollars a student — the point where the budget stops
   reaching everybody — and it is flat within 2 % between 700 and 800.
2. **Buy the cheap thing first.** Caregiver messaging is pinned at its cap at every
   budget the state might have. It is 6 % of the per-student cost and about a third of the
   per-student effect.
3. **The pilot's 90 minutes loses by about 13,800 cohort reading points** — 19 million
   dollars at the department's own valuation — with posterior probability 0.997. Not
   because 90 minutes does not work: it works on 4,900 children instead of 12,000.
4. **What could overturn it.** The advantage would have to be overstated by its *whole
   amount* before the pilot wins; `tipping_point` walks the bias grid and finds the flip
   at 100 % of the estimate. The response family does not change the allocation at this
   budget, only the value it promises — the two families part company only at three times
   the money. The **eligible count** does change the allocation, from 72 weekly minutes at
   6,000 students to 16 at 24,000, and is the number to re-check before the money moves.
5. **What this trial cannot say.** The grade-band breakdown the program office asked for
   is not estimable from a school-randomized design; `realize` refused it in notebook 3
   and the memo says so rather than reporting a number nobody can defend.